In [11]:
%pip install pandas transformers deepparse huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [12]:
import torch
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
    torch.set_default_device('cuda')
elif torch.xpu.is_available():
    print("XPU - available devices:")
    for i in range(torch.xpu.device_count()):
        print(f"  {i}: {torch.xpu.get_device_name(i)}")
    torch.set_default_device('xpu')
else: print("WARNING: Running on CPU")
print(f"Torch version: {torch.__version__}, Device: {torch.get_default_device()}")

XPU - available devices:
  0: Intel(R) Arc(TM) 130V GPU (16GB)
Torch version: 2.10.0+xpu, Device: xpu:0


In [13]:
from huggingface_hub import notebook_login
notebook_login()

In [14]:
import pandas as pd

def confusion_metrics(preds : pd.DataFrame, labels : pd.DataFrame, ignore_trash_columnds = False):
    # Drop meta columns included in the preds dataframe
    preds = preds.drop(columns=["fullResponse", "fullAddress", "parseError"], errors="ignore")
    
    # Confusion matrix is analyzed for each field of each row
    # false positives means any positve prediction that differs from the ground truth
    #(ground truth may be empty for that field)
    false_positives = 0
    # false negatives means any field present in ground truth but not correctly predicted
    #(prediction may be empty for that field)
    false_negatives = 0
    # true positives means any field correctly predicted
    #(exact match between prediction and ground truth values)
    true_positives = 0
    # true negatives means any field correctly not predicted
    #(absent in both prediction and ground truth)
    true_negatives = 0


    if not ignore_trash_columnds:
        # labels that should not have been predicted at all
        trash_columns = [col for col in preds.columns if col not in labels.columns]
        false_positives += preds[trash_columns].notna().sum().sum()
    for col in labels.columns:
        if col not in preds.columns:
            # all ground truth spans in this column are false negatives
            false_negatives += labels[col].notna().sum()
        else:
            false_positives += (preds[col].notna() & (labels[col] != preds[col])).sum()
            true_negatives += (preds[col].isna() & labels[col].isna()).sum()
            true_positives += (preds[col].notna() & (labels[col] == preds[col])).sum()
            false_negatives += (labels[col].notna() & (labels[col] != preds[col])).sum()
    
    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    f1 = 2 * (precision * recall) / (precision + recall)
    accuracy = (true_positives + true_negatives) / (
        true_positives + false_positives + false_negatives + true_negatives)
    return {
        "recall" : recall,
        "precision" : precision,
        "f1" : f1,
        "accuracy" : accuracy
    }

COLS_TO_PREDICT = [
    "HouseNumber",
    "StreetName",
    "City",
    "State",
    "Country"
]

EXCEPT_COUNTRY = COLS_TO_PREDICT[:-1]

In [15]:
import transformers
from transformers import pipeline

class LlamaAddressSegmentationModel:
    def __init__(self, model_name, prompt, output_parser = None, batch_size=32):
        tokenizer = transformers.AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", padding_side='left')
        self.pipe = pipeline("text-generation", model=model_name, batch_size=batch_size, tokenizer=tokenizer)
        self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id[0]
        self.prompt = prompt
        self.output_parser = output_parser or (lambda x : x)

    def segment_addresses(self, addresses : list[str]) -> str:
        messages = [[
            {"role": "user", "content": self.prompt.format(address=address)}
        ] for address in addresses]
        result = self.pipe(messages)
        responses = [self.output_parser(r[0]["generated_text"][1]["content"]) for r in result]
        return responses

def extract_json_block(model_response : str):
    limit_chars = [('{', '}'), ('[', ']'), ('"', '"')]
    json_str = model_response
    parts = model_response.split("```")
    if len(parts) >= 2: # single code block or malformed code block
        json_str = parts[1]
    elif len(parts) >= 3:
        for part in parts:
            part = part.strip()
            if part.startswith("json") or any(part.startswith(lim[0]) and part.endswith(lim[1]) for lim in limit_chars):
                json_str = part
                break
    if json_str.startswith("json"):
        json_str = json_str[4:].strip()
    return json_str

In [16]:
import json

class JSONObjectParser:
    def __call__(self, response: str) -> dict:
        try:
            json_str = extract_json_block(response)
            obj = json.loads(json_str)
            for k in obj.keys():
                assert k not in ["fullResponse", "model-fullAddress", "parseError"], f"Key collision with reserved field: {k}"
                # It is plausible that the model just echoes back the full address in a field named "fullAddress"
                # But that would collide with our own field named "fullAddress" that we add later
                if k == "fullAddress":
                    obj["model-fullAddress"] = obj["fullAddress"] # avoid collision but keep the wrongfully generated field
            obj["fullResponse"] = response
            data = obj
        except Exception as e:
            data = {"fullResponse": response, "parseError": str(e)}
        return data

class JSONTuplesParser:
    def __call__(self, response: str) -> dict:
        try:
            json_str = extract_json_block(response)
            tuples = json.loads(json_str)
            data = {}
            for part, ptype in tuples:
                if isinstance(part, str) and part.strip() == "":
                    continue
                assert ptype not in ["fullResponse", "model-fullAddress", "parseError"], f"Key collision with reserved field: {ptype}"
                if ptype == "fullAddress":
                    data["model-fullAddress"] = data["fullAddress"] # avoid collision but keep the wrongfully generated field
                else:
                    collision_value = data.get(ptype)
                    if collision_value is None:
                        data[ptype] = part
                    else:
                        data[ptype] = collision_value + "___" + part
            data["fullResponse"] = response
        except Exception as e:
            data = {"fullResponse": response, "parseError": str(e)}
        return data

In [17]:
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv")
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_test.csv")

EXAMPLES = [
    ("Berlin, Alexanderplatz 1, 10178", 
     "{{ \"City\": \"Berlin\", \"StreetName\": \"Alexanderplatz\", \"HouseNumber\": \"1\"}}"),
    ("Braunschweig, Uferstr. 25", 
     "{{ \"City\": \"Braunschweig\", \"StreetName\": \"Uferstr.\", \"HouseNumber\": \"25\"}}"),
    ("808 Westend Avenue, New York 25, N.Y.",
        "{{ \"StreetName\": \"Westend Avenue\", \"HouseNumber\": \"808\", "
        "\"City\": \"New York\", \"State\": \"N.Y.\"}}"),
]

PROMPTS = [
    "Segment addresses into their components.\n"
    "Output only a JSON object with the following keys: " + ", ".join(COLS_TO_PREDICT) + ". "
    "Do not include fields that cannot be determined and do not try to guess their values. "
    "For example, if the address is simply \"Berlin\" then the field \"Country\" should be null."
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word. "
    "Consider the following examples:" +
    ''.join(f"Address: {adr}\n```json\n{ex}\n```\n" for adr, ex in EXAMPLES) +
    "Now segment the following address:\n{address}",

    "Annotate each component in the following address: {address}\n"
    "Consider the component types houseNumber, streetName, city, state, country, other. "
    "Not all addresses will contain all component types and you may not guess the missing ones. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as \"stadt\" for city, \"Straße\" for street or its abbreviation \"Str.\" "
    "These terms may occur as a suffix to another word. "
    "Output only a JSON list of [component, type] tuples. "
    "For example, the address \"Berlin, Alexanderplatz 1, 10178\" should be annotated as:\n"
    "```json\n[ [\"Berlin\", \"city\"], [\"Alexanderplatz\", \"street\"], [\"1\", \"streetNumber\"], [\"10178\", \"other\"] ]\n```",
]

In [ ]:

import time

model_configs = [
    {
        "model_name": "meta-llama/Llama-3.2-1B-Instruct",
        "prompt": 0,
        "output_parser": JSONObjectParser()
    },
    {
        "model_name": "meta-llama/Llama-3.2-3B-Instruct",
        "prompt": 0,
        "output_parser": JSONObjectParser()
    },
    {
        "model_name": "meta-llama/Llama-3.2-1B-Instruct",
        "prompt": 1,
        "output_parser": JSONTuplesParser()
    },
    {
        "model_name": "meta-llama/Llama-3.2-3B-Instruct",
        "prompt": 1,
        "output_parser": JSONTuplesParser()
    },
]

preds_per_config = []

def eval(dataset):
    global preds_per_config
    model_results = []
    for config in model_configs:
        model_name = config["model_name"]
        prompt = PROMPTS[config["prompt"]]
        output_parser = config["output_parser"]
        print(f"Loading model {model_name}...")
        model = LlamaAddressSegmentationModel(model_name, prompt, output_parser)
        print(f"Segmenting addresses...")
        start = time.monotonic()
        responses = model.segment_addresses(dataset["FullAddress"].tolist())
        print("Parsing model responses...")
        preds_df = pd.DataFrame(responses)
        preds_per_config.append(pd.concat([dataset["FullAddress"], preds_df], axis=1))
        print("Computing confusion metrics...")
        metrics = confusion_metrics(preds_df, dataset[COLS_TO_PREDICT])
        deltatime = time.monotonic() - start
        metrics["deltatime"] = deltatime
        metrics["rate"] = len(dataset) / deltatime

        model_results.append(metrics)
        print(f"Results for model {model_name}: {metrics}")


    results_df = pd.DataFrame(model_results, index = pd.MultiIndex.from_tuples(
        [(config["model_name"], config["prompt"]) for config in model_configs],
    ))
    return results_df

model_statistics = eval(bzkopen_val)

model_statistics

Loading model meta-llama/Llama-3.2-1B-Instruct...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Segmenting addresses...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

Parsing model responses...
Computing confusion metrics...
Results for model meta-llama/Llama-3.2-1B-Instruct: {'recall': 0.5934959349593496, 'precision': 0.4, 'f1': 0.47790507364975454, 'accuracy': 0.5507042253521127, 'deltatime': 82.375, 'rate': 1.6024279210925645}
Loading model meta-llama/Llama-3.2-3B-Instruct...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Segmenting addresses...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
for model_config, pred in zip(model_configs, preds_per_config):
    new_index = pd.MultiIndex.from_product([
        [model_config["model_name"]],
        [model_config["prompt"]],
        pred.index
    ])
    pred.index = new_index

preds_per_config = pd.concat(preds_per_config)

,FullAddress,City,StreetName,HouseNumber,fullResponse,Country,State,Town
0,"Regensburg, Königstr. 2/I",Regensburg,Königstr.,2/I,"For the address: Regensburg, Königstr. 2/I\n\n...",NaN,NaN,NaN
1,Dortmund,Dortmund,NaN,NaN,"Based on the provided examples, the address ""D...",NaN,NaN,NaN
2,Jöhlingen/Krs. Durlach/Baden.,Jöhlingen,NaN,None,"Based on the given address ""Jöhlingen/Krs. Dur...",Baden,Krs. Durlach,NaN
3,"8 Burlington Road, Manchester 20/England.",Manchester,Burlington Road,8,"Based on the given address, here is the segmen...",England,20,NaN
4,Leer/Ostfriesland,Leer,NaN,NaN,"For the address ""Leer/Ostfriesland"", the segme...",NaN,Ostfriesland,NaN
...,...,...,...,...,...,...,...,...
127,Sosnowice/Polen,NaN,Sosnowice,NaN,"Based on the address ""Sosnowice/Polen"", we can...",Polen,NaN,NaN
128,2114-79 St. Jackson Heights. N.Y. USA,Jackson Heights,St.,2114-79,"```json\n{\n ""HouseNumber"": ""2114-79"",\n ""St...",USA,N.Y.,NaN
129,Losone CSR,NaN,CSR,Losone,"Based on the provided examples, it appears tha...",NaN,NaN,NaN
130,Rum.,NaN,Rum.,NaN,"Since the address ""Rum."" contains a suffix, we...",Romania,NaN,NaN
